In [ ]:
# import TSS data 
import pandas as pd, numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score

# --- Load data (your paths) ---
pos = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/positive_values/merged_TTS_500bp_20bins.csv")
neg = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negative_values/ATAC_negatives_2k_features_TTS_only.csv")

pos["label"] = 1
neg["label"] = 0
data = pd.concat([pos, neg], ignore_index=True)

# Build features (drop label + TSS_coord if present)
X_df = data.drop(columns=[c for c in ["label", "TTS_coord"] if c in data.columns], errors="ignore")
X_df = X_df.select_dtypes(include=["number"])
feature_names = X_df.columns.tolist()

print("Total features:", X_df.shape[1])

print(X_df)

In [ ]:
import re

# select biology features 
GROUP_A = {"H2A.Z","H2B.Z","H3K27ac","H3K9ac","H3K18ac","H3K4me3","H2A.Zac","ATAC"}   # bins 05–15
GROUP_B = {"H3","H3K36me3","H3K4me","MNase","H3K27me","H3K4me2","H3R17me2","H3K18me","H3K4me1"}  # bins 05–15

pat = re.compile(r"^(?P<mark>.+?)_TTS_bin(?P<bin>\d+)$")

def keep(col: str) -> bool:
    m = pat.match(col)
    if not m:
        return False
    mark = m.group("mark")
    b = int(m.group("bin"))
    return (mark in GROUP_A and 5 <= b <= 15) or (mark in GROUP_B and 5 <= b <= 15)

# Select and (optionally) sort by mark then bin
selected_cols = [c for c in X_df.columns if keep(c)]
selected_cols = sorted(selected_cols, key=lambda c: (pat.match(c).group("mark"),
                                                     int(pat.match(c).group("bin"))))

X_sel = X_df[selected_cols].copy()
feature_names = selected_cols
print(f"Selected {len(selected_cols)}/{X_df.shape[1]} features")
print(X_sel)

# y from the 'label' column, aligned to X_sel's index
y = data.loc[X_sel.index, "label"].astype(int)

# quick sanity checks
assert len(y) == len(X_sel)
print(y.value_counts())

In [ ]:
# ANNOVA and de-correlate
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import f_classif

# --- CONFIG (tune as needed) ---
Q_FDR_CUTOFF = 0.05      # BH-FDR threshold for p-values
ETA2_MIN     = 0.01      # minimum effect size
CORR_THRESH  = 0.95      # max allowed absolute correlation between kept features
TOPK_FALLBACK = 300      # if FDR/eta2 keep too few, fall back to top-K by F

# X_sel: your pre-filtered feature matrix (numeric DataFrame)
# y:     labels (0/1 or multi-class)

# 1) Median-impute for stats
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X_sel), columns=X_sel.columns, index=X_sel.index)

# 2) ANOVA (per-feature)
F, p = f_classif(X_imp.values, y.values)
F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
p = np.nan_to_num(p, nan=1.0, posinf=1.0, neginf=1.0)

# 2b) BH-FDR (with safe fallback if statsmodels unavailable)
try:
    from statsmodels.stats.multitest import multipletests
    q = multipletests(p, method="fdr_bh")[1]
except Exception:
    # simple BH implementation
    m = len(p)
    order = np.argsort(p)
    ranked = np.empty_like(order)
    ranked[order] = np.arange(1, m+1)
    q = p * m / ranked
    # enforce monotonicity
    q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
    q = np.empty_like(q_sorted)
    q[order] = q_sorted

# effect size eta^2
k = y.nunique(); N = len(y)
eta2 = (F * (k - 1)) / (F * (k - 1) + (N - k) + 1e-12)

anova_tbl = pd.DataFrame({
    "feature": X_sel.columns,
    "F": F, "p": p, "q": q, "eta2": eta2
}).sort_values("F", ascending=False).reset_index(drop=True)

# 3) Keep by FDR & effect size (or fallback to top-K)
keep_anova = anova_tbl.query("q < @Q_FDR_CUTOFF and eta2 >= @ETA2_MIN")["feature"].tolist()
if len(keep_anova) == 0:
    keep_anova = anova_tbl.head(min(TOPK_FALLBACK, len(anova_tbl)))["feature"].tolist()

X_anova = X_imp[keep_anova]

# 4) De-correlate greedily by ANOVA rank
#    Traverse features (high→low F) and keep a feature only if it's not highly correlated with any already kept.
corr = X_anova.corr().abs()
ordered_feats = anova_tbl[anova_tbl["feature"].isin(keep_anova)]["feature"].tolist()  # sorted by F desc

kept = []
for f in ordered_feats:
    if not kept:
        kept.append(f)
        continue
    # check correlation with all already kept
    if (corr.loc[f, kept].abs() <= CORR_THRESH).all():
        kept.append(f)

print(f"ANOVA-kept: {len(keep_anova)}  |  After de-correlation: {len(kept)}")

# 5) Final matrix with de-correlated features
X_final = X_sel[kept].copy()

# Save artifacts
anova_tbl.to_csv("anova_table.csv", index=False)
pd.Series(keep_anova, name="anova_keep").to_csv("features_after_anova.csv", index=False)
pd.Series(kept, name="final_features").to_csv("features_after_decorrelation.csv", index=False)

# Optional peek
anova_tbl.head(10)


In [ ]:
# === Random Forest with embedded selection, nested GroupKFold by chromosome ===
import numpy as np, pandas as pd, joblib, os
from statistics import mode
from collections import Counter

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier

# -------------------------------------------------------------------------
# Inputs expected:
#   - X_final: DataFrame with your 153 selected features
#   - y:       Series/array of 0/1 labels aligned to X_final
#   - data:    original DataFrame containing a chromosome column for grouping
# -------------------------------------------------------------------------

# 0) Get chromosome groups
CANDIDATES = ["chr", "chrom", "chromosome", "chrom_id"]
GROUP_COL = next((c for c in CANDIDATES if c in data.columns), None)
if GROUP_COL is None:
    raise ValueError(f"No chromosome column found. Add one of {CANDIDATES} to `data`.")

groups = data.loc[X_final.index, GROUP_COL].astype(str)

# 1) Set outer/inner grouped CV sizes
n_chr = groups.nunique()
OUTER_SPLITS = min(5, max(2, n_chr))    # need at least 2 unique chromosomes
INNER_SPLITS = min(3, max(2, n_chr - 1))

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
inner_cv = GroupKFold(n_splits=INNER_SPLITS)

print(f"GroupCV by '{GROUP_COL}': outer={OUTER_SPLITS}, inner={INNER_SPLITS}, groups={n_chr} chromosomes")

# 2) Pipeline: log1p -> impute -> embedded RF selector -> RF classifier
log1p_tf = FunctionTransformer(lambda X: np.log1p(np.clip(X, 0, None)), validate=False)

# Base RF; we’ll tune a few keys in the grid
rf_base = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",   # handle imbalance
    n_jobs=-1,
    random_state=42
)

pipe = Pipeline([
    ("log1p", log1p_tf),
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ),
        threshold=-np.inf,        # rank purely by RF impurity importance
        max_features=None         # tuned below
    )),
    ("clf", rf_base),
])

# 3) Hyperparameter grid (compact but effective)
param_grid = {
    "sel__max_features": [10, 25, 50, 83]  # None = keep all 153
    # "clf__max_depth": [None, 10, 20],
    # "clf__min_samples_leaf": [1, 2, 5],
    # "clf__max_features": ["sqrt", "log2", 0.5],    # 0.5 means 50% of features
}

# 4) Nested GroupKFold
fold_ap, fold_roc, chosen_K, selected_masks = [], [], [], []

for i, (tr, te) in enumerate(outer_cv.split(X_final, y, groups), start=1):
    X_tr, X_te = X_final.iloc[tr], X_final.iloc[te]
    y_tr, y_te = y.iloc[tr], y.iloc[te]
    g_tr, g_te = groups.iloc[tr], groups.iloc[te]

    gs = GridSearchCV(
        pipe, param_grid=param_grid, cv=inner_cv,
        scoring="average_precision", n_jobs=-1, verbose=0,
        refit=True, error_score="raise"
    )
    gs.fit(X_tr, y_tr, groups=g_tr)

    best_est = gs.best_estimator_
    y_score = best_est.predict_proba(X_te)[:, 1]

    ap  = average_precision_score(y_te, y_score)
    roc = roc_auc_score(y_te, y_score)
    fold_ap.append(ap); fold_roc.append(roc)
    chosen_K.append(gs.best_params_["sel__max_features"])

    # record selection mask
    mask = best_est.named_steps["sel"].get_support()
    selected_masks.append(mask)

    print(f"[Outer {i}/{OUTER_SPLITS}] K={gs.best_params_['sel__max_features']} | AP={ap:.4f} | ROC={roc:.4f}")

print("\n===== Grouped (by chromosome) Nested CV with RandomForest =====")
print(f"AP  mean±sd: {np.mean(fold_ap):.4f} ± {np.std(fold_ap):.4f}")
print(f"ROC mean±sd: {np.mean(fold_roc):.4f} ± {np.std(fold_roc):.4f}")
print(f"Chosen K per outer fold: {chosen_K}")

# 5) Selection stability across outer folds
sel_freq = np.mean(np.vstack(selected_masks), axis=0)
freq_series = pd.Series(sel_freq, index=X_final.columns).sort_values(ascending=False)
os.makedirs("rf_groupcv_out", exist_ok=True)
freq_series.to_csv("rf_groupcv_out/selection_frequency.csv")
print("Top 20 most frequently selected features:\n", freq_series.head(20))

In [ ]:
cv_results = pd.DataFrame(gs.cv_results_)
print(cv_results[["param_sel__max_features", "mean_test_score"]])


In [ ]:
from sklearn.pipeline import Pipeline
import numpy as np, pandas as pd, joblib, os
from statistics import mode
from collections import Counter

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
# put this at top-level (not inside another function/cell)
def log1p_nonneg(X):
    import numpy as np
    return np.log1p(np.clip(X, 0, None))

from sklearn.preprocessing import FunctionTransformer
log1p_tf = FunctionTransformer(log1p_nonneg, validate=False)
final_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=1,
            max_features="sqrt", class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        threshold=-np.inf, max_features=75
    )),
    ("clf", RandomForestClassifier(
        n_estimators=600, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=42
    )),
])


In [ ]:
import numpy as np, pandas as pd, joblib, os
from statistics import mode
from collections import Counter

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
# put this at top-level (not inside another function/cell)
def log1p_nonneg(X):
    import numpy as np
    return np.log1p(np.clip(X, 0, None))

from sklearn.preprocessing import FunctionTransformer
log1p_tf = FunctionTransformer(log1p_nonneg, validate=False)


final_pipe.fit(X_final, y)
import joblib
joblib.dump(final_pipe, "rf_groupcv_out/rf_groupcv_pipeline.joblib")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# --- Split or use held-out test set ---
# If you already have train/test, use those.
# If not, here we show example with a 20% test split:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_final, y, test_size=0.2,
                                          stratify=y, random_state=42)

# Fit on training only
final_pipe.fit(X_tr, y_tr)

# --- ROC curve ---
y_score = final_pipe.predict_proba(X_te)[:, 1]
fpr, tpr, _ = roc_curve(y_te, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve on Test Set (Gene End Classifier)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- Confusion matrix ---
y_pred = final_pipe.predict(X_te)   # default threshold = 0.5
cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix on Test Set (Gene End Classifier)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_feature_importances(pipe, X, top_n=20, out_file=None):
    # Step 1: extract importances as Series
    mask = pipe.named_steps["sel"].get_support()
    selected_features = X.columns[mask]
    importances = pipe.named_steps["clf"].feature_importances_
    fi = pd.Series(importances, index=selected_features).sort_values(ascending=False)

    # Step 2: select top N
    fi_top = fi.head(top_n)

    # Step 3: plot
    plt.figure(figsize=(8, 6))
    sns.barplot(x=fi_top.values, y=fi_top.index, palette="viridis")
    plt.title(f"Top {top_n} Feature Importances (Gene End Classifier)")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    
    if out_file:
        plt.savefig(out_file, dpi=200)
        print(f"Saved -> {out_file}")
    else:
        plt.show()
    
    return fi

# Example usage:
fi = plot_feature_importances(final_pipe, X_final, top_n=20,
                              out_file="rf_groupcv_out/feature_importances_top20.png")
